## Exercises on Actor–Critic Methods

These paper-and-pencil exercises reinforce Chapter 09: the advantage function and its centring property, the $n$-step actor–critic target (with the **correct** baseline $V(S_t)$), the TD error as a one-step advantage, the bias–variance trade-off between Monte Carlo and bootstrapped critics, Generalised Advantage Estimation (GAE) and its derivation as a discounted sum of TD errors, the handling of termination and truncation in a batched rollout, and the normalisation of advantages. They build on the policy-gradient foundations of Chapter 08. Notation: actor parameters $\theta$, critic $V(\cdot;\phi)$, true advantage $a_\pi(s,a)$ and its estimates $A(S_t,A_t)$, $A^{(n)}_t$, TD error $\delta_t$, terminal flag $d_{t+1}$; inside a rollout, $G_t$ abbreviates the bootstrapped $n$-step return $G_{t:t+n}$ of step $t$.

### Exercise 9.1 — The advantage function and its centring property

In a state $s$ the action-values are $q_\pi(s,\cdot)=(2,\,0,\,1)$ for three actions, and the policy is $\pi(\cdot\mid s)=(0.5,\,0.3,\,0.2)$.

1. Compute the state value $v_\pi(s)$ and the advantages $a_\pi(s,a)=q_\pi(s,a)-v_\pi(s)$.
2. Verify that the policy-weighted advantage is zero, and explain why this matters.

**Step 1 — State value and advantages.** $\ v_\pi(s)=\sum_a\pi(a\mid s)\,q_\pi(s,a) = 0.5(2)+0.3(0)+0.2(1)=1.2.$

$\displaystyle a_\pi(s,\cdot) = q_\pi(s,\cdot)-v_\pi(s) = (2,0,1)-1.2 = (0.8, -1.2, -0.2).$

**Step 2 — Centring.** $\ \sum_a\pi(a\mid s)\,a_\pi(s,a) = 0.5(0.8)+0.3(-1.2)+0.2(-0.2) = 0.0.$

This is no accident: $\sum_a\pi\, a_\pi = \sum_a\pi\, q_\pi - v_\pi\sum_a\pi = v_\pi - v_\pi = 0$. Under the policy, advantages average to zero — positive for better-than-average actions, negative for worse.

**Key concept**

The advantage measures *how much better than the policy's average* an action is. Because it is centred at zero, using it as the weight in the policy gradient increases the probability of above-average actions and decreases below-average ones, giving a lower-variance signal than the raw return.

### Exercise 9.2 — The $n$-step actor–critic target

From state $S_t$ the agent collects a 3-step segment with rewards $R_{t+1},R_{t+2},R_{t+3}=0,0,1$ and then bootstraps on the critic. With $\gamma=0.9$, critic estimates $V(S_t)=0.5$ and $V(S_{t+3})=0.7$:

1. Compute the 3-step return $G_{t:t+3}$.
2. Compute the $n$-step advantage $A^{(3)}_t = G_{t:t+3}-V(S_t)$ and state the critic's regression target.

**Step 1 — $n$-step return** (three real rewards, then bootstrap on $V(S_{t+3})$):

$\displaystyle G_{t:t+3} = R_{t+1}+\gamma R_{t+2}+\gamma^2 R_{t+3}+\gamma^3 V(S_{t+3}) = 0+0+0.81(1)+0.729(0.7) = 1.3203.$

**Step 2 — Advantage and critic target.** The advantage subtracts the value of the **current** state $S_t$ (the baseline is $V(S_t)$, *not* $V(S_{t+n})$):

$\displaystyle A^{(3)}_t = G_{t:t+3} - V(S_t) = 1.3203 - 0.5 = 0.8203.$

The **actor** uses $A^{(3)}_t$ to weight $\nabla_\theta\log\pi_\theta(A_t\mid S_t)$; the **critic** regresses $V(S_t)$ toward the target $G_{t:t+3}=1.3203$ (loss $(G_{t:t+3}-V(S_t))^2$).

**Key concept**

An actor–critic replaces the Monte Carlo return with an $n$-step **bootstrapped** return: fewer real rewards plus the critic's estimate of the rest. The bootstrap introduces bias (the critic is imperfect) but cuts variance. The baseline subtracted is always the value of the state where the action was taken, $V(S_t)$.

### Exercise 9.3 — Generalised Advantage Estimation (GAE)

Along a trajectory the one-step TD errors are $\delta_t=R_{t+1}+\gamma V(S_{t+1})-V(S_t)$, computed as $(\delta_0,\delta_1,\delta_2)=(0.2,\,0.5,\,1.0)$. With $\gamma=0.9$ and $\lambda=0.5$:

1. Explain why $\delta_t$ is itself a (one-step) advantage estimate.
2. Compute the GAE advantage $A^{\text{GAE}}_0 = \sum_{l\ge0}(\gamma\lambda)^l\,\delta_{t+l}$ at $t=0$.
3. Give the $\lambda=0$ and $\lambda=1$ limits.

**Step 1 — $\delta_t$ as an advantage.** $\ \delta_t = R_{t+1}+\gamma V(S_{t+1}) - V(S_t)$ is the one-step return minus the baseline $V(S_t)$ — exactly a one-step estimate of $A(S_t,A_t)$ (an unbiased estimate of the advantage when $V=v_\pi$).

**Step 2 — GAE at $t=0$** with discount $\gamma\lambda=0.9(0.5)=0.45$:

$\displaystyle A^{\text{GAE}}_0 = \delta_0 + (\gamma\lambda)\delta_1 + (\gamma\lambda)^2\delta_2 = 0.2 + 0.45(0.5) + 0.2025(1.0) = 0.6275.$

**Step 3 — Limits.**
- $\lambda=0$: $\ A^{\text{GAE}}_0 = \delta_0 = 0.2$ — the **one-step** (high-bias, low-variance) advantage.
- $\lambda=1$: $\ A^{\text{GAE}}_0 = \sum_l \gamma^l\delta_l = 0.2+0.9(0.5)+0.81(1.0) = 1.46$ — the full-trajectory (low-bias, high-variance) advantage, which coincides with the **Monte Carlo** advantage when the episode ends at $S_3$ with $V(S_3)=0$.

**Key concept**

GAE is the advantage analogue of the $\lambda$-return: an exponentially weighted average of $n$-step advantage estimates, controlled by $\lambda$. It lets us dial continuously between the low-variance one-step critic ($\lambda=0$) and the unbiased Monte Carlo estimate ($\lambda=1$), typically choosing $\lambda\approx0.95$.

### Exercise 9.4 — Bias and variance of critic targets (integrative)

From state $s$ the dynamics are: a deterministic step with reward $R_{t+1}=0$ into $s_1$, then from $s_1$ a reward $R_{t+2}$ that is $+2$ or $0$ with equal probability (mean $1$), then termination; $\gamma=1$. The true values are $v(s_1)=1$ and $v(s)=1$.

1. Compute the mean and variance of the **Monte Carlo** target $G_t$ used to update $V(s)$.
2. Compute the mean and variance of the **one-step (bootstrapped)** target $R_{t+1}+\gamma V(s_1)$, first with an exact critic $V(s_1)=1$, then with a biased critic $V(s_1)=0.8$.
3. Summarise the bias–variance trade-off.

**Step 1 — Monte Carlo target.** $\ G_t = R_{t+1}+R_{t+2} = 0 + R_{t+2} \in\{0,2\}$, each with prob. $0.5$.

$\displaystyle \mathbb{E}[G_t] = 1 \ (\text{unbiased, since }v(s)=1), \qquad \mathrm{Var}[G_t] = \tfrac12(0^2+2^2)-1^2 = 2-1 = 1.$

**Step 2 — One-step bootstrapped target** $R_{t+1}+\gamma V(s_1) = 0 + V(s_1)$ (a *constant*, since $R_{t+1}$ is deterministic and $V(s_1)$ is fixed):

- Exact critic $V(s_1)=1$: target $=1$, $\ \mathbb{E}=1$ (unbiased), $\ \mathrm{Var}=0$.
- Biased critic $V(s_1)=0.8$: target $=0.8$, $\ \mathbb{E}=0.8$ (**bias** $=-0.2$), $\ \mathrm{Var}=0$.

**Step 3 — Trade-off.** The MC target is **unbiased** but carries the full variance of the stochastic future reward ($\mathrm{Var}=1$). The bootstrapped target has **zero variance** here (it replaces the random future by the critic's number) but is **only as correct as the critic**: an imperfect $V(s_1)$ injects bias ($-0.2$). Actor–critic methods accept a little bias to buy a large variance reduction; GAE (Exercise 9.3) tunes exactly where on this spectrum to sit.

**Key concept**

Bootstrapping trades variance for bias. Monte Carlo returns are unbiased but noisy; one-step critic targets are stable but biased by critic error. The whole design space of actor–critic methods — $n$-step returns, GAE, target networks — is about managing this trade-off.

### Exercise 9.5 — The $n$-step advantage telescopes into TD errors

Along a trajectory the critic predicts $V(S_0),V(S_1),V(S_2),V(S_3) = 1.0,\,0.8,\,0.5,\,0.2$ and the rewards are $R_1,R_2,R_3 = 0.48,\,0.85,\,1.32$, with $\gamma=0.9$.

1. Compute the one-step TD errors $\delta_0,\delta_1,\delta_2$.
2. Prove that $A^{(2)}_t = \delta_t+\gamma\,\delta_{t+1}$ and, in general, $A^{(n)}_t=\sum_{l=0}^{n-1}\gamma^l\delta_{t+l}$.
3. Verify numerically that the 3-step advantage $A^{(3)}_0$ computed from its definition equals $\delta_0+\gamma\delta_1+\gamma^2\delta_2$.

**Step 1 — TD errors** $\delta_t = R_{t+1}+\gamma V(S_{t+1})-V(S_t)$:

$\displaystyle \delta_0 = 0.48+0.9(0.8)-1.0 = 0.20, \qquad \delta_1 = 0.85+0.9(0.5)-0.8 = 0.50, \qquad \delta_2 = 1.32+0.9(0.2)-0.5 = 1.00.$

(These are the TD errors of Exercise 9.3.)

**Step 2 — Telescoping.** Start from the definition and add and subtract $\gamma V(S_{t+1})$:

$\displaystyle A^{(2)}_t = R_{t+1}+\gamma R_{t+2}+\gamma^2 V(S_{t+2})-V(S_t) = \underbrace{\big[R_{t+1}+\gamma V(S_{t+1})-V(S_t)\big]}_{\delta_t} + \gamma\underbrace{\big[R_{t+2}+\gamma V(S_{t+2})-V(S_{t+1})\big]}_{\delta_{t+1}}.$

Repeating the insertion of $\pm\gamma^{l}V(S_{t+l})$ for $l=1,\dots,n-1$ gives $A^{(n)}_t=\sum_{l=0}^{n-1}\gamma^l\delta_{t+l}$: every intermediate value appears once with a plus and once with a minus sign and cancels, leaving only $-V(S_t)$ at the start and $+\gamma^nV(S_{t+n})$ at the end.

**Step 3 — Numerical check.** From the definition:

$\displaystyle A^{(3)}_0 = R_1+\gamma R_2+\gamma^2R_3+\gamma^3V(S_3)-V(S_0) = 0.48+0.765+1.0692+0.1458-1.0 = 1.46.$

From the TD errors: $\ \delta_0+\gamma\delta_1+\gamma^2\delta_2 = 0.20+0.45+0.81 = 1.46.\ \checkmark$ (This is also the $\lambda=1$ value found in Exercise 9.3.)

**Key concept**

Every $n$-step advantage is a **partial discounted sum of the same sequence of TD errors**: one term for the one-step estimate, all terms for the Monte Carlo estimate. This is what makes it possible to mix all the $n$-step estimates at once, and to compute them for a whole rollout in a single backward pass.

### Exercise 9.6 — GAE as an exponentially weighted average

GAE is defined as the average $A^{\text{GAE}}_t = (1-\lambda)\sum_{n\ge1}\lambda^{n-1}A^{(n)}_t$ of the $n$-step advantages.

1. Using Exercise 9.5, show that $A^{\text{GAE}}_t = \sum_{l\ge0}(\gamma\lambda)^l\,\delta_{t+l}$.
2. With the data of Exercise 9.5 ($\gamma=0.9$) and $\lambda=0.5$, assume that the critic is exact from $S_3$ onward, so that all TD errors $\delta_l$ with $l\ge3$ vanish and $A^{(n)}_0=A^{(3)}_0$ for all $n\ge3$. Compute $A^{\text{GAE}}_0$ as a weighted average of $A^{(1)}_0,A^{(2)}_0,A^{(3)}_0$ and compare with the TD-error form.

**Step 1 — Exchange the sums.** Substituting $A^{(n)}_t=\sum_{l=0}^{n-1}\gamma^l\delta_{t+l}$, the TD error $\delta_{t+l}$ appears in every $A^{(n)}_t$ with $n\ge l+1$. Its total coefficient is

$\displaystyle (1-\lambda)\,\gamma^l\sum_{n=l+1}^{\infty}\lambda^{n-1} = (1-\lambda)\,\gamma^l\,\frac{\lambda^{l}}{1-\lambda} = (\gamma\lambda)^l,$

using the geometric series $\sum_{n\ge l+1}\lambda^{n-1} = \lambda^l/(1-\lambda)$. Hence $A^{\text{GAE}}_t=\sum_{l\ge0}(\gamma\lambda)^l\delta_{t+l}$: a discounted sum of TD errors with discount $\gamma\lambda$.

**Step 2 — Weighted average.** The $n$-step advantages at $t=0$ are $A^{(1)}_0=\delta_0=0.20$, $A^{(2)}_0=\delta_0+\gamma\delta_1=0.65$, $A^{(3)}_0=1.46$. The weights $(1-\lambda)\lambda^{n-1}$ with $\lambda=0.5$ are $0.5,\,0.25,\,0.125,\dots$; since all $A^{(n)}_0$ with $n\ge3$ equal $A^{(3)}_0$, their weights add up to $0.125+0.0625+\dots=0.25$:

$\displaystyle A^{\text{GAE}}_0 = 0.5(0.20)+0.25(0.65)+0.25(1.46) = 0.1+0.1625+0.365 = 0.6275.$

TD-error form (Exercise 9.3): $\ 0.20+0.45(0.50)+0.2025(1.00) = 0.6275.\ \checkmark$

**Key concept**

GAE is to the advantage what the $\lambda$-return is to the value: an exponentially weighted average of all the $n$-step estimates, which collapses to a **discounted sum of TD errors with discount $\gamma\lambda$**. $\lambda$ moves continuously from the one-step TD error ($\lambda=0$, low variance, high bias) to the Monte Carlo advantage ($\lambda=1$, unbiased, high variance).

### Exercise 9.7 — Targets in a batched rollout: termination vs. truncation

An A2C agent runs $K=2$ environments for $n=3$ steps with $\gamma=0.9$. Every reward is $1$. At the end of the rollout the critic predicts $V=5$ for the last state of both environments. In environment A no episode ends. In environment B the episode **terminates** at the second step (the flag $d=1$ is recorded at $t=1$), the environment is reset, and the third step belongs to a new episode.

1. Compute the three $n$-step returns of environment A.
2. Compute the three returns of environment B.
3. Suppose instead that the episode in B had been **truncated** by the time limit at $t=1$, with the critic predicting $V=4$ for the final state. How does the rollout record this, and what are the returns? What would go wrong if truncation were treated as termination?

**Step 1 — Environment A** (recursion $G_t=R_{t+1}+\gamma(1-d_{t+1})G_{t+1}$ started from the bootstrap value $5$):

$\displaystyle G_2 = 1+0.9(5) = 5.5, \qquad G_1 = 1+0.9(5.5) = 5.95, \qquad G_0 = 1+0.9(5.95) = 6.355.$

**Step 2 — Environment B** (the terminal flag at $t=1$ cuts the recursion):

$\displaystyle G_2 = 1+0.9(5) = 5.5, \qquad G_1 = 1+0.9\,(1-1)\,G_2 = 1, \qquad G_0 = 1+0.9(1) = 1.9.$

The return of the new episode's first step ($G_2$) does not leak into the old episode: $G_1$ is just the terminal reward, and $G_0$ sees two rewards and no bootstrap.

**Step 3 — Truncation.** The state reached at $t=1$ is *not* terminal: it has a future value, but the environment is reset and that state never becomes the start of a stored transition. The rollout therefore bootstraps immediately, **adding $\gamma V=0.9(4)=3.6$ to the reward** of that step, $R'_2 = 1+3.6 = 4.6$, and still marking the step as the end of the episode ($d=1$). The returns become

$\displaystyle G_2 = 5.5, \qquad G_1 = 4.6, \qquad G_0 = 1+0.9(4.6) = 5.14.$

Treating the truncation as a termination would give $G_1=1$ and $G_0=1.9$ instead: the critic would be trained to believe that a perfectly healthy state (the pole still up when the time limit struck) is worth $1$ rather than about $4.6$, a **systematic bias** that propagates backwards to every state visited near the time limit, and, through the advantages, to the actor.

**Key concept**

In a batched rollout the terminal flag does two jobs: it stops the return recursion at episode boundaries, so that episodes in the same column do not contaminate each other, and it distinguishes **termination** (no future value) from **truncation** (bootstrap on the final state). Confusing the two corrupts the critic's targets exactly where the state is fine.

### Exercise 9.8 — Normalising the advantages

A rollout produced the advantage estimates $A=(3,\,-1,\,5,\,1)$. Many implementations **normalise** the advantages of each batch to zero mean and unit standard deviation before computing the policy loss.

1. Compute the normalised advantages $\hat A_i = (A_i-\bar A)/\sigma_A$.
2. Which of the two operations (subtracting the mean, dividing by the standard deviation) leaves the expected policy gradient unchanged, and which one changes it? What is the practical benefit of each?

**Step 1 — Normalise.** $\ \bar A = (3-1+5+1)/4 = 2$; deviations $(1,-3,3,-1)$; $\sigma_A=\sqrt{(1+9+9+1)/4}=\sqrt5=2.236$:

$\displaystyle \hat A = (0.447,\ -1.342,\ 1.342,\ -0.447).$

The positive advantages stay positive and the negative stay negative, but the batch is now centred and has a fixed scale.

**Step 2 — Which is harmless.** Subtracting a constant $\bar A$ from every advantage is a **baseline** (a constant one, computed from the batch): by the zero-expected-score identity it leaves the expected gradient unchanged, up to a small finite-batch bias, since $\bar A$ is itself computed from the sampled actions, while it reduces variance, exactly as in Chapter 08. Dividing by $\sigma_A$ **rescales** the gradient of the batch by $1/\sigma_A$: the direction is unchanged but the step length is not, so this is not a pure variance reduction but an **adaptive step size**, large when the advantages are small and small when they are large (for instance after a surprising return). It makes the actor's learning rate and the entropy coefficient $\beta$ insensitive to the scale of the rewards, which varies by orders of magnitude across environments, at the cost of a small bias when the batch is tiny.

**Key concept**

Advantage normalisation is two tricks in one: a per-batch baseline (unbiased variance reduction) and a per-batch rescaling of the step (a stabiliser that decouples hyperparameters from the reward scale). Together with gradient clipping it is one of the standard safeguards of actor-critic implementations, and it reappears in PPO (Chapter 11).